# This file is for data cleansing only
The data set we use here are:
1. Air quality data from Open data Bristol
2. EVs sale data from GOV.uk

### Step1 : Import data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df_evs = pd.read_csv('C:/Users/Admin/Desktop/DSMP/Raw data/Bristol car sales data -EV population.csv')
df_aq = pd.read_csv("C:/Users/Admin/Desktop/DSMP/Raw data/air-quality-data-continuous.csv", sep=";")

df_evs

### Step2 : Clean data
### 2.1 Clean EVs data

In [ ]:
# drop uneccessary columns
df_evs = df_evs.drop(columns=['ONS Code [note 6]','ONS Sort [note 6]','Units'],axis=1)

# Filter data in columns
df_evs = df_evs.loc[df_evs['Keepership'] == 'Total']
df_evs = df_evs.loc[df_evs['ONS Geography [note 6]'] == '   Bristol, City of']
df_evs = df_evs.loc[df_evs['Fuel'] == 'Battery electric']

# Remove comma in data and change city name
df_evs = df_evs.replace('   Bristol, City of', 'Bristol', regex=True)
df_evs = df_evs.replace(',', '', regex=True)

df_evs

In [ ]:
# Get a list of the old column names
old_columns = list(df_evs.columns[3:])

# Melt the DataFrame using the old column names as the values in the "Date" column
melted_evs = df_evs.melt(id_vars=['Fuel', 'Keepership', 'ONS Geography [note 6]'], var_name='Date', value_name='Volume', value_vars=old_columns)

melted_evs

In [ ]:
# Extract year and quarter from "Date" column
melted_evs["Year"] = melted_evs["Date"].str.split(" ").str[0]
melted_evs["Quarter"] = melted_evs["Date"].str.split(" ").str[1]

# Convert quarter to month
quarter_dict = {"Q1": "03", "Q2": "06", "Q3": "09", "Q4": "12"}
melted_evs["Month"] = melted_evs["Quarter"].map(quarter_dict)

# Convert year, month and day to datetime
melted_evs["Date"] = pd.to_datetime(melted_evs["Year"] + melted_evs["Month"] + "01", format="%Y%m%d") + pd.offsets.QuarterEnd()

# Calculate start date of quarter
melted_evs["Quarter Start"] = melted_evs["Date"] - pd.offsets.MonthBegin(3)

# Drop unnecessary columns
melted_evs = melted_evs.drop(["Year", "Quarter", "Month"], axis=1)

# Rename the columns
melted_evs = melted_evs.rename(columns={"ONS Geography [note 6]": "City"})
melted_evs = melted_evs.rename(columns={"Date": "Quarter End"})

# Drop Keepership column
melted_evs = melted_evs.drop(columns='Keepership',axis=1)

# Re columns index
melted_evs = melted_evs.reindex(columns=['Fuel', 'City', 'Volume', 'Quarter Start', 'Quarter End'])
melted_evs


In [ ]:
# change data frame name
df_evs = melted_evs
df_evs.to_csv('DSMP_EVs.csv')

### 2.2 Clean Air quality data

In [ ]:
# Check data data info
df_aq.info()
df_aq

In [ ]:
# Drop unneccessary data
df_aq = df_aq.loc[:, ['Date Time', 'NOx', 'Location', 'geo_point_2d']]

# Split data into lat and long
df_aq[["Latitude", "Longitude"]] = df_aq["geo_point_2d"].str.split(",", expand=True)

# Drop geo_point_2d 
df_aq = df_aq.drop(columns='geo_point_2d', axis=1)

# Convert the "Date Time" column to datetime type
df_aq["Date Time"] = pd.to_datetime(df_aq["Date Time"])

# Extract the date and time components
df_aq["Date"] = df_aq["Date Time"].dt.date
df_aq["Time"] = df_aq["Date Time"].dt.time

# Drop the original "Date Time" column
# df_aq.drop(columns=["Date Time"], inplace=True)

df_aq

In [ ]:
# Check the data
df_aq.info()

In [ ]:
# Drop Null value of NOx
df_aq = df_aq.dropna()
df_aq.to_csv('DSMP_AQ.csv')

### Step3: Merge both data frame by following the condition below
1. If the "Date" column from df_aq is in the range of date between "Quarter start" and "Quarter End" from df_evs, we will merge all column of df_evs  with df_aq
2. If the "Date" column from df_aq is not match in the range of date between "Quarter start" and "Quarter End" from df_evs, we will drop that row

In [ ]:
import pandas as pd

# Convert the date columns to datetime objects
df_evs['Quarter Start'] = pd.to_datetime(df_evs['Quarter Start'])
df_evs['Quarter End'] = pd.to_datetime(df_evs['Quarter End'])
df_aq['Date'] = pd.to_datetime(df_aq['Date'])

# Merge the dataframes based on the date ranges
df_merged = pd.merge_asof(df_aq.sort_values('Date'), df_evs.sort_values('Quarter Start'),
                          left_on='Date', right_on='Quarter Start')

# Drop any rows from the merged dataframe where the date is not within the quarter date range
mask = (df_merged['Quarter Start'] <= df_merged['Date']) & (df_merged['Quarter End'] >= df_merged['Date'])
df_merged = df_merged[mask]

df_merged


In [ ]:
# Re columns index
df_merged = df_merged.reindex(columns=['Fuel', 'City', 'Date', 'Time', 'Quarter Start', 'Quarter End',
                                       'Volume', 'NOx', 'Location', 'Latitude', 'Longitude', 'Date Time'])

# Rename column
df_merged = df_merged.rename(columns={"Volume": "EVs Volume"})

df_merged

In [ ]:
df_merged.to_csv('DSMP_EVs_AQ.csv')